# Building RAG system to store and retrieve Pokemon Data

## Introduction
As silly Pokemon nerds, we want to collect info about Pokemon easily and from a good source. The problem is the API is not that handy for querying, and sites elsewhere are not always accurate or reliable.</br></br>
Our goal is to create a system to query for Pokemon information. We do this in 3 parts:
<ol><li>Generate a queryable RAG (Retrieval-Augmented Generation) system of reliable Pokemon data.</li>
<li>Create a web search tool to look for answers from the net.</li>
<li>Build agents and a supervisor with LangGraph which would select the right tool for the job and provide results.</li>
</ol>

In [1]:
# Imports, parameters and general-purpose helper functions
import os, json, re, requests, concurrent.futures
from pathlib import Path
from tqdm import tqdm
from pydantic import BaseModel, Field
from typing import Optional, List
from langchain_core.tools import tool
import numpy as np, requests
import faiss, json
from typing import List, Dict, Any, Optional, Literal
import nltk
import pandas as pd

# Set to true to rebuild database of Pokemon info
BUILD_INDEX = False

# General-purpose helper functions

CACHE = Path("./poke_cache"); CACHE.mkdir(exist_ok=True)
BASE  = "https://pokeapi.co/api/v2"

# --- region <-> generation mapping (authoritative) ---
REGION_TO_GEN = {
    "kanto":"generation-i","johto":"generation-ii","hoenn":"generation-iii",
    "sinnoh":"generation-iv","unova":"generation-v","kalos":"generation-vi",
    "alola":"generation-vii","galar":"generation-viii","paldea":"generation-ix",
}
GEN_TO_REGION = {v:k for (k,v) in REGION_TO_GEN.items()}

def _session():
    s = requests.Session()
    s.headers.update({"User-Agent":"poke-rag/1.0"})
    return s

def _cache_get(name):
    f = CACHE/f"{name}.json"
    return json.load(open(f)) if f.exists() else None

def _cache_put(name, obj):
    f = CACHE/f"{name}.json"
    with open(f,"w") as fp: json.dump(obj, fp)

def get_json(url, name=None, timeout=15):
    if name:
        c = _cache_get(name)
        if c is not None: return c
    s = _session()
    r = s.get(url, timeout=timeout)
    r.raise_for_status()
    data = r.json()
    if name: _cache_put(name, data)
    return data

def list_all(resource):
    # e.g. resource="pokemon-species"
    key = f"index_{resource}"
    cached = _cache_get(key)
    if cached: return cached["results"]
    url = f"{BASE}/{resource}?limit=2000"
    data = get_json(url)
    _cache_put(key, data)
    return data["results"]

def pull_many(index, namer, workers=16):
    s = _session()
    def pull(rec):
        r = s.get(rec["url"], timeout=15); r.raise_for_status()
        data = r.json()
        _cache_put(namer(data), data)
        return data
    out=[]
    with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as ex:
        for d in tqdm(ex.map(pull, index), total=len(index), desc="pull"):
            out.append(d)
    return out


## RAG System Development
### Query, load and index data into Vector Database

In [2]:
# Query and load data into documents

def move_effect_en(move) -> str:
    # pick English effect; prefer short_effect
    en = next((e for e in move.get("effect_entries", []) if e["language"]["name"]=="en"), None)
    txt = (en.get("short_effect") or en.get("effect") or "").replace("\f"," ").strip() if en else ""
    # substitute $effect_chance with the numeric value if present
    chance = move.get("effect_chance")
    if chance is not None:
        txt = txt.replace("$effect_chance", str(chance))
    # normalize whitespace
    return re.sub(r"\s+"," ", txt)

def clean_en(entries, key="language"):
    for e in entries:
        lang = e.get(key, {}).get("name")
        if lang == "en":
            # PokeAPI flavor/effect texts have \n/\f and strange spacing
            return re.sub(r"\s+", " ", (e.get("flavor_text") or e.get("short_effect") or e.get("effect") or "").replace("\f"," ")).strip()
    return ""

def default_types_for_species(species_json):
    for v in species_json.get("varieties", []):
        if v.get("is_default") and v.get("pokemon"):
            p = get_json(v["pokemon"]["url"], name=f"pokemon_{v['pokemon']['name']}")
            return [t["type"]["name"] for t in sorted(p["types"], key=lambda x: x["slot"])]
    return []

def move_doc(m):
    mid = m["id"]; name=m["name"]
    type_ = (m.get("type") or {}).get("name")
    dmg = (m.get("damage_class") or {}).get("name")
    power, acc, pp = m.get("power"), m.get("accuracy"), m.get("pp")
    eff_text = move_effect_en(m)
    main = {
        "id": mid, "kind":"move","name":name,"type":type_,"damage_class":dmg,
        "power":power,"accuracy":acc,"pp":pp, "effect_text": eff_text   
    }
    text = f"Move {name} — Type: {type_}. Class: {dmg}. Power: {power}. Accuracy: {acc}. PP: {pp}. Effect: {eff_text}"
    return {"id": f"move:{mid}", "name": name, "text": text, "meta": main}

def species_doc(s):
    name = s["name"]; sid = s["id"]
    shape = (s.get("shape") or {}).get("name")
    color = (s.get("color") or {}).get("name")
    genus = next((g["genus"] for g in s.get("genera",[]) if g["language"]["name"]=="en"), None)
    flavor = clean_en(s.get("flavor_text_entries", []))
    gen = (s.get("generation") or {}).get("name")
    types = default_types_for_species(s)                # ← NEW

    meta = {
        "id": sid, "kind":"species", "name": name,
        "shape": shape, "color": color, "genus": genus,
        "is_mythical": s["is_mythical"], "is_legendary": s["is_legendary"],
        "generation": gen, "types": types               # ← NEW
    }
    text = (f"{name.capitalize()} — {genus or 'Pokemon'}. Types: {', '.join(types) or 'unknown'}. "
            f"Shape: {shape}. Color: {color}. Legendary: {s['is_legendary']}. Mythical: {s['is_mythical']}. "
            f"Generation: {gen}. Flavor: {flavor}")
    return {"id": f"species:{sid}", "name": name, "text": text, "meta": meta}

def pokemon_doc(p, species_lookup):
    name = p["name"]; pid = p["id"]
    types = [t["type"]["name"] for t in sorted(p["types"], key=lambda x:x["slot"])]
    abilities = [a["ability"]["name"] for a in p["abilities"]]
    moves = [m["move"]["name"] for m in p["moves"][:12]]
    h_m, w_kg = p["height"]/10.0, p["weight"]/10.0
    sp = species_lookup.get(name)
    main = {
        "id": pid, "kind":"pokemon", "name": name, "types": types,
        "height_m": h_m, "weight_kg": w_kg, "abilities": abilities, "moves": moves,
        "is_mythical": sp.get("is_mythical") if sp else None, "is_legendary": sp.get("is_legendary") if sp else None,
        "generation": sp.get("generation") if sp else None
    }
    text = f"{name.capitalize()} — Types: {', '.join(types)}. Abilities: {', '.join(abilities)}. " \
           f"Size: {h_m} m, {w_kg} kg. Common moves: {', '.join(moves)}. " \
           f"Legendary: {main['is_legendary']}. Mythical: {main['is_mythical']}."
    return {"id": f"pokemon:{pid}", "name": name, "text": text, "meta": main}


def ability_doc(a):
    aid=a["id"]; name=a["name"]
    effect = clean_en(a.get("effect_entries", []))
    short  = clean_en(a.get("effect_entries", []))
    main = {"id":aid,"kind":"ability","name":name}
    text = f"Ability {name} — {short or effect}"
    return {"id": f"ability:{aid}", "name": name, "text": text, "meta": main}

def type_doc(t):
    tid = t["id"]; name=t["name"]
    rel = t.get("damage_relations", {})
    def names(xs): return ", ".join(e["name"] for e in xs) or "none"
    text = (f"Type {name} — double_damage_to: {names(rel.get('double_damage_to',[]))}; "
            f"double_damage_from: {names(rel.get('double_damage_from',[]))}; "
            f"half_damage_to: {names(rel.get('half_damage_to',[]))}; "
            f"half_damage_from: {names(rel.get('half_damage_from',[]))}; "
            f"no_damage_to: {names(rel.get('no_damage_to',[]))}; "
            f"no_damage_from: {names(rel.get('no_damage_from',[]))}.")
    return {"id": f"type:{tid}", "name": name, "text": text, "meta": {"id":tid, "kind":"type","name":name}}

def generation_doc(g):
    gid=g["id"]; name=g["name"]; region=(g.get("main_region") or {}).get("name")
    text=f"{name} — main region: {region}"
    return {"id": f"generation:{gid}", "name": name, "text": text, "meta":{"id":gid,"kind":"generation","region":region}}

def region_doc(r):
    rid=r["id"]; name=r["name"]
    locs = [l["name"] for l in r.get("locations", [])[:50]]
    text=f"Region {name} — notable locations: {', '.join(locs)}"
    return {"id": f"region:{rid}", "name": name, "text": text, "meta":{"id":rid,"kind":"region"}}

if BUILD_INDEX:
    # Query the Pokemon API and load the corpus into docs
    # Index species (fast), pokemon (heavy), moves/abilities/types/generations/regions (medium)
    species_idx = list_all("pokemon-species")
    species = pull_many(species_idx, namer=lambda d: f"species_{d['id']}")
    species_map = {s["name"]: {"is_mythical":s["is_mythical"],"is_legendary":s["is_legendary"],"generation":(s.get("generation") or {}).get("name")} for s in species}

    moves = pull_many(list_all("move")[:400], namer=lambda d: f"move_{d['id']}")           # cap first to keep it snappy
    abilities = pull_many(list_all("ability")[:300], namer=lambda d: f"ability_{d['id']}")
    types = pull_many(list_all("type"), namer=lambda d: f"type_{d['id']}")
    gens  = pull_many(list_all("generation"), namer=lambda d: f"generation_{d['id']}")
    regions = pull_many(list_all("region"), namer=lambda d: f"region_{d['id']}")

    docs = []
    docs += [species_doc(s) for s in species]
    docs += [move_doc(m) for m in moves]
    docs += [ability_doc(a) for a in abilities]
    docs += [type_doc(t) for t in types]
    docs += [generation_doc(g) for g in gens]
    docs += [region_doc(r) for r in regions]

    # (Optional later) Pokémon proper (heavy): uncomment when ready
    # pokemon = pull_many(list_all("pokemon")[:300], namer=lambda d: f"pokemon_{d['id']}")
    # docs += [pokemon_doc(p, species_map) for p in pokemon]


In [ ]:
# Index and embed the data and set up the FAISS tool
EMBED_MODEL = "nomic-embed-text"  # or "bge-m3"
OLLAMA_URL = "http://localhost:11434/api/embeddings"

def embed(texts):
    vecs = []
    for t in texts:
        r = requests.post(OLLAMA_URL, json={"model": EMBED_MODEL, "prompt": t}, timeout=60)
        r.raise_for_status()
        v = r.json()["embedding"]
        vecs.append(np.array(v, dtype=np.float32))
    return np.vstack(vecs)

def normalize(v):
    n = np.linalg.norm(v, axis=1, keepdims=True) + 1e-9
    return v / n

def build_faiss(docs, path="datasets/poke_faiss"):
    ''' Sets up the FAISS (Facebook AI Similarity Search) tool to perform similarity search and vector clustering'''
    texts = [d["text"] for d in docs]
    X = normalize(embed(texts))
    index = faiss.IndexFlatIP(X.shape[1])
    index.add(X)
    faiss.write_index(index, f"{path}.index")
    with open(f"{path}.meta.json","w") as fp:
        json.dump(docs, fp)
    return path

def load_faiss(path="datasets/poke_faiss"):
    index = faiss.read_index(f"{path}.index")
    meta  = json.load(open(f"{path}.meta.json"))
    return index, meta

if BUILD_INDEX:
    # Perform the ETL -- do the FAISS indexing and metadata generation
    path = build_faiss(docs)
    index, meta = load_faiss(path)

### Plug into RAG using LangChain / LangGraph

In [4]:

OLLAMA_URL = os.environ.get("OLLAMA_EMBED_URL", "http://localhost:11434/api/embeddings")
EMBED_MODEL = os.environ.get("OLLAMA_EMBED_MODEL", "nomic-embed-text")

def _embed_ollama(texts: List[str]) -> np.ndarray:
    vecs = []
    for t in texts:
        r = requests.post(OLLAMA_URL, json={"model": EMBED_MODEL, "prompt": t}, timeout=60)
        r.raise_for_status()
        vecs.append(np.array(r.json()["embedding"], dtype=np.float32))
    X = np.vstack(vecs)
    # cosine via inner product → normalize
    X /= (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
    return X

class PokeRAG:
    def __init__(self, index_path: str = "poke_faiss"):
        idx_file = f"{index_path}.index"
        meta_file = f"{index_path}.meta.json"
        if not (os.path.exists(idx_file) and os.path.exists(meta_file)):
            raise FileNotFoundError(
                f"Missing index/meta. Expected {idx_file} and {meta_file}. "
                "Build them with your ETL/index step first."
            )
        self.index = faiss.read_index(idx_file)
        with open(meta_file, "r") as fp:
            self.meta = json.load(fp)

    def _filter(self, d: Dict[str, Any], *, 
                kind: Optional[str], mythical_only: bool,
                types: Optional[List[str]], region: Optional[str],
                generation: Optional[str], shape: Optional[str]) -> bool:
        m = d.get("meta", {})
        if kind and m.get("kind") != kind:
            return False
        if mythical_only and not m.get("is_mythical"):
            return False
        if types:
            doc_types = {t.lower() for t in (m.get("types") or [])}
            if not set(t.lower() for t in types) <= doc_types:
                return False
        if region and (m.get("region") or m.get("main_region")) != region:
            return False
        if generation and (m.get("generation")) != generation:
            return False
        if shape and ((m.get("shape") or "").lower() != shape.lower()):   
            return False
        return True

    def search(self, query: str, k: int = 8, *, 
               kind: Optional[str] = None, mythical_only: bool = False,
               types: Optional[List[str]] = None, region: Optional[str] = None,
               generation: Optional[str] = None, shape: Optional[str] = None) -> List[Dict[str, Any]]:
        qv = _embed_ollama([query])
        D, I = self.index.search(qv, max(k * 5, k))  # overfetch then filter
        hits = []
        for i, score in zip(I[0], D[0]):
            if i < 0: 
                continue
            d = self.meta[i]
            if self._filter(d, kind=kind, mythical_only=mythical_only,
                            types=types, region=region, generation=generation, shape=shape):
                out = {
                    "id": d["id"],
                    "name": d.get("name"),
                    "kind": d["meta"].get("kind"),
                    "score": float(score),
                    "snippet": (d.get("text","")[:240] + "…") if len(d.get("text","")) > 240 else d.get("text",""),
                    "meta": d.get("meta", {}),
                }
                hits.append(out)
                if len(hits) >= k:
                    break
        return hits

    # Convenience wrapper returning a dict (nice for tools)
    def search_dict(self, **kwargs) -> Dict[str, Any]:
        return {"results": self.search(**kwargs)}


In [5]:
# Previous method of loading Pokemon features. Use this if the RAG results are not helpful.
# Query features for pokemon
# Convert size to meters and kg. (PokeAPI gives height in decimeters and weight in hectograms.)
GEN_TO_REGION = {
    "generation-i": "Kanto",
    "generation-ii": "Johto",
    "generation-iii": "Hoenn",
    "generation-iv": "Sinnoh",
    "generation-v": "Unova",
    "generation-vi": "Kalos",
    "generation-vii": "Alola",
    "generation-viii": "Galar",
    "generation-ix": "Paldea",
}

def _meters(decimeters:int) -> float: return round(decimeters / 10.0, 2)
def _kg(hectograms:int) -> float:     return round(hectograms / 10.0, 1)

def _size_class(height_m: float) -> str:
    if height_m < 0.8: return "small"
    if height_m <= 1.5: return "medium"
    return "large"

def _latest_version_group(details):
    """Pick the newest version_group present in the learnset."""
    # crude order: add more as needed
    order = ["red-blue","yellow","gold-silver","crystal","ruby-sapphire","emerald",
             "diamond-pearl","platinum","black-white","black-2-white-2","x-y",
             "omega-ruby-alpha-sapphire","sun-moon","ultra-sun-ultra-moon",
             "sword-shield","scarlet-violet"]
    rank = {k:i for i,k in enumerate(order)}
    best = None; best_rank = -1
    for d in details:
        vg = d.get("version_group",{}).get("name")
        if vg in rank and rank[vg] > best_rank:
            best_rank = rank[vg]; best = vg
    return best

def _choose_key_moves(poke_json, types):
    stabs = {t["type"]["name"] for t in poke_json["types"]}
    # filter level-up moves in the latest version group
    moves = []
    for m in poke_json["moves"]:
        md = [d for d in m["version_group_details"] if d["move_learn_method"]["name"] == "level-up"]
        if not md: continue
        vg = _latest_version_group(md)
        if not vg: continue
        moves.append(m["move"]["name"])

    # rank by: STAB > known/iconic names > power (requires per-move fetch)
    ranked = []
    for mv in moves[:40]:  # cap network calls
        try:
            mj = _get(f"{BASE}/move/{mv}")
            power = mj.get("power") or 0
            mtype = mj["type"]["name"]
            stab = 1 if mtype in stabs else 0
            iconic = 1 if mv in {"tackle","quick-attack","yawn","thunderbolt","shadow-ball","water-gun"} else 0
            ranked.append((stab, iconic, power, mj["name"]))
        except Exception:
            continue
    ranked.sort(reverse=True)
    out = [mv for _,_,_,mv in ranked[:3]]
    # Title-case nicely
    return [s.replace("-", " ").title() for s in out] or ["Tackle"]

def fetch_pokemon_features(name: str):
    nid = name.strip().lower()
    p = _get(f"{BASE}/pokemon/{nid}")
    s = _get(f"{BASE}/pokemon-species/{nid}")

    types = [t["type"]["name"].title() for t in sorted(p["types"], key=lambda x: x["slot"])]
    height_m = _meters(p["height"])
    weight_kg = _kg(p["weight"])
    size = _size_class(height_m)
    region = GEN_TO_REGION.get(s["generation"]["name"], "Unknown")

    shape = (s.get("shape") or {}).get("name")          # e.g., 'quadruped', 'fish', 'upright', ...
    color = (s.get("color") or {}).get("name")          # e.g., 'pink', 'blue', ...
    genus = next((g["genus"] for g in s.get("genera", [])
                  if g["language"]["name"] == "en"), "")
    genus = genus.replace("Pokémon", "").strip().lower()
    isNoun = lambda pos: pos[:2] == 'NN'
    tokenized = nltk.word_tokenize(genus)
    genus = [word for (word, pos) in nltk.pos_tag(tokenized) if isNoun (pos)]
    if genus:
        genus = genus[0]
    else:
        genus = ""

    flavor_en = " ".join(ft["flavor_text"].replace("\n", " ").replace("\f", " ")
                         for ft in s.get("flavor_text_entries", [])
                         if ft["language"]["name"] == "en")


    # Top-listed ability is fine; you can prefer non-hidden
    abilities = [a["ability"]["name"].replace("-", " ").title() for a in p["abilities"]]
    if abilities:
        # prefer non-hidden first
        abilities.sort(key=lambda a: 1 if "Hidden" in a else 0)

    key_moves = _choose_key_moves(p, types)

    return {
        "name": name.title(),
        "region": region,
        "types": types,                # e.g., ["Water","Psychic"]
        "size": size,                  # small / medium / large
        "height_m": height_m,
        "weight_kg": weight_kg,
        "shape": shape,                # e.g., 'quadruped', 'fish', 'upright', ...
        "color": color,                # e.g., 'pink', 'blue', ...
        "genus": genus,                # e.g., 'Seed Pokémon'
        "flavor_text": flavor_en,      # concatenated English flavor text entries
        "is_legendary": s.get("is_legendary", False) or s.get("is_mythical", False),
        "is_mythical": s.get("is_mythical", False),
        "key_moves": key_moves,        # 2–3 iconic/typed moves
        "ability": abilities[0] if abilities else None,
        "sources": {
            "pokemon": f"{BASE}/pokemon/{nid}",
            "species": f"{BASE}/pokemon-species/{nid}"
        }
    }


In [6]:
# tool_poke_rag.py  — patched: adds mode="list" + clean views
from typing import Optional, List, Literal, Dict, Any
from pydantic import BaseModel, Field
from langchain_core.tools import tool

POKE_RAG = PokeRAG(index_path="poke_faiss")

def _roman(n:int)->str:
    r = ["","I","II","III","IV","V","VI","VII","VIII","IX","X",
         "XI","XII","XIII","XIV","XV","XVI","XVII","XVIII","XIX","XX"]
    return r[n] if 0 <= n < len(r) else str(n)

def _cap(s:str) -> str:
    return (s or "").replace("-", " ").title()

def _table_for(hits, kind_lc):
    if (kind_lc or (hits and hits[0].get("kind"))) == "generation":
        cols = ["Generation", "Region"]
        rows = []
        for h in hits:
            gid = h.get("meta",{}).get("id")
            reg = _cap(h.get("meta",{}).get("region"))
            rows.append([f"Generation {_roman(gid)}", reg])
    else:
        cols = ["Name", "Kind"]
        rows = [[_cap(h.get("name")), h.get("kind")] for h in hits]
    # markdown + very simple HTML (fallback-safe)
    md = "| " + " | ".join(cols) + " |\n|"+ "|".join(["---"]*len(cols)) + "|\n" + \
         "\n".join("| " + " | ".join(map(str,row)) + " |" for row in rows)
    html = "<table>" + \
           "<thead><tr>" + "".join(f"<th>{c}</th>" for c in cols) + "</tr></thead>" + \
           "<tbody>" + "".join("<tr>" + "".join(f"<td>{c}</td>" for c in row) + "</tr>" for row in rows) + \
           "</tbody></table>"
    return cols, rows, md, html

class PokeRAGArgs(BaseModel):
    # query / filters
    query: str = Field("", description="Natural-language question or keywords.")
    k: int = Field(8, ge=1, le=9999)
    kind: Optional[str] = Field(None, description="species|pokemon|move|ability|type|generation|region")
    mythical_only: bool = False
    types: Optional[List[str]] = None
    region: Optional[str] = None
    generation: Optional[str] = None
    shape: Optional[str] = Field(None, description="Filter species by PokéAPI shape slug, e.g., 'quadruped', 'fish'.")
    require_all_types: bool = Field(False, description="If true, doc must include ALL listed types (AND).")
    # formatting
    view: Literal["raw","names","pairs","text","table","dataframe","html"] = "raw"
    distinct: bool = Field(True, description="Deduplicate by name in formatted views.")
    # retrieval mode
    mode: Literal["search","list"] = Field("search", description="list = deterministic scan by kind; search = vector search")

@tool("poke_rag_search", args_schema=PokeRAGArgs)
def poke_rag_search(
    query: str = "",
    k: int = 8,
    kind: Optional[str] = None,
    mythical_only: bool = False,
    types: Optional[List[str]] = None,
    region: Optional[str] = None,
    generation: Optional[str] = None,
    shape: Optional[str] = None,
    require_all_types: bool = False,
    view: str = "raw",
    distinct: bool = True,
    mode: str = "search",
) -> Dict[str, Any]:
    """Search local Poké RAG (FAISS + metadata). Use mode='list' for complete enumerations (e.g., generations)."""

    # ---- shared filter ----
    want_types = {t.lower() for t in (types or [])}
    reg_lc = (region or "").lower()
    gen_lc = (generation or "").lower()
    kind_lc = (kind or "").lower()

    def _passes(d: Dict[str, Any]) -> bool:
        m = d.get("meta", {})
        if kind and (m.get("kind","").lower() != kind_lc): return False
        if mythical_only and not m.get("is_mythical"): return False
        if types:
            doc_types = {t.lower() for t in (m.get("types") or [])}
            if require_all_types:
                if not (want_types <= doc_types): return False  # AND
            else:
                if not (want_types & doc_types): return False    # OR
        if region and ((m.get("region") or m.get("main_region") or "").lower() != reg_lc): return False
        if generation and ((m.get("generation") or "").lower() != gen_lc): return False
        if shape and ((m.get("shape") or "").lower() != shape.lower()): return False 
        return True

    # ---- retrieval ----
    if mode == "list" and kind:
        # Deterministic scan of all docs of this kind (no vector search).
        items = [d for d in POKE_RAG.meta if (d.get("meta",{}).get("kind","").lower() == kind_lc)]
        items = [d for d in items if _passes(d)]
        # sort: by numeric meta.id if present, else by name
        def sort_key(d):
            mid = d.get("meta",{}).get("id")
            nm  = d.get("name") or ""
            return (0, mid) if isinstance(mid, int) else (1, nm)
        items.sort(key=sort_key)
        hits = [{
            "id": d["id"],
            "name": d.get("name"),
            "kind": d.get("meta",{}).get("kind"),
            "score": 1.0,
            "snippet": (d.get("text","")[:240] + "…") if len(d.get("text",""))>240 else d.get("text",""),
            "meta": d.get("meta",{}),
        } for d in items[:k]]
    else:
        # Vector search first, then filter; top up with list-mode if kind is set and results are sparse.
        hits = POKE_RAG.search(
            query=query or kind or "pokemon",
            k=max(k*5, k),  # overfetch for filtering
            kind=None, mythical_only=False, types=None, region=None, generation=None,
            shape=shape
        )
        hits = [h for h in hits if _passes(h)]
        # Top-up to k with deterministic list of same kind (if provided)
        if kind and len(hits) < k:
            seen = {(h.get("name") or "").lower() for h in hits}
            pool = [d for d in POKE_RAG.meta
                    if (d.get("meta",{}).get("kind","").lower() == kind_lc)
                    and _passes(d)
                    and ((d.get("name") or "").lower() not in seen)]
            def sort_key(d):
                mid = d.get("meta",{}).get("id")
                nm  = d.get("name") or ""
                return (0, mid) if isinstance(mid, int) else (1, nm)
            pool.sort(key=sort_key)
            for d in pool[:(k - len(hits))]:
                hits.append({
                    "id": d["id"], "name": d.get("name"),
                    "kind": d.get("meta",{}).get("kind"),
                    "score": 0.0,
                    "snippet": (d.get("text","")[:240] + "…") if len(d.get("text",""))>240 else d.get("text",""),
                    "meta": d.get("meta",{}),
                })
        # trim to k
        hits = hits[:k]

    # ---- formatting ----
    if not hits or view == "raw":
        return {"results": hits}

    # stable sort again for deterministic formatting
    def sort_key_fmt(h):
        mid = h.get("meta",{}).get("id")
        nm  = h.get("name") or ""
        return (0, mid) if isinstance(mid, int) else (1, nm)
    hits.sort(key=sort_key_fmt)

    # dedupe by name if requested
    if distinct:
        seen=set(); tmp=[]
        for h in hits:
            nm = (h.get("name") or "").lower()
            if nm in seen: continue
            seen.add(nm); tmp.append(h)
        hits = tmp

    if view == "names":
        items = []
        for h in hits:
            if (kind_lc or h.get("kind")) == "generation":
                gid = h.get("meta",{}).get("id")
                items.append(f"Generation {_roman(gid) if isinstance(gid,int) else _cap(h.get('name'))}")
            else:
                items.append(_cap(h.get("name")))
        return {"items": items}

    if view == "pairs":
        rows = []
        for h in hits:
            knd = (kind_lc or h.get("kind"))
            if knd == "generation":
                gid = h.get("meta",{}).get("id")
                reg = _cap(h.get("meta",{}).get("region"))
                rows.append({"generation": f"Generation {_roman(gid) if isinstance(gid,int) else _cap(h.get('name'))}",
                             "region": reg})
            elif knd in {"species","pokemon"}:
                rows.append({"name": _cap(h.get("name")),
                             "types": [_cap(t) for t in (h.get("meta",{}).get("types") or [])],
                             "shape": _cap(h.get("meta",{}).get("shape") or ""), })
            else:
                rows.append({"name": _cap(h.get("name")), "kind": knd})
        return {"rows": rows}

    if view == "text":
        lines=[]
        for h in hits:
            knd = (kind_lc or h.get("kind"))
            if knd == "generation":
                gid = h.get("meta",{}).get("id")
                reg = _cap(h.get("meta",{}).get("region"))
                lines.append(f"- Generation {_roman(gid)} — {reg}")
            elif knd in {"species","pokemon"}:
                types_list = [_cap(t) for t in (h.get("meta",{}).get("types") or [])]
                lines.append(f"- {_cap(h.get('name'))}" + (f" — {', '.join(types_list)}" if types_list else ""))
            else:
                lines.append(f"- {_cap(h.get('name'))}")
        return {"text": "\n".join(lines)}

    if view in ("table","dataframe","html"):
        cols, rows, md, html = _table_for(hits, kind_lc)
        out = {"columns": cols, "rows": rows}
        if view == "table":
            out["markdown"] = md
        if view == "html":
            out["html"] = html
        if view == "dataframe":
            out["table"] = out  # explicit table payload for UIs
        return out

    # fallback
    return {"results": hits}


class PokemonArgs(BaseModel):
    pokemon_name: str

@tool("web_pokemon_features", args_schema=PokemonArgs)
def web_pokemon_features_tool(args: PokemonArgs) -> dict:
    '''Collect standard features (name, region, types, size, ...) from an api'''
    return fetch_pokemon_features(args.pokemon_name)


In [7]:
# graph_setup.py
from langgraph.prebuilt import create_react_agent
from langchain_ollama import ChatOllama
# (plus your other tools like web_pokemon_features_tool, sd image tool, etc.)

llm = ChatOllama(model="gpt-oss:20b", temperature=0.2)

TOOLS = [poke_rag_search, web_pokemon_features_tool]  # add others as needed

research_agent = create_react_agent(
    model=llm,
    tools=TOOLS,
    name="research_agent",
    prompt=(
        "You research Pokémon facts using local RAG first. "
        "Prefer poke_rag_search for types, generations, regions, mythicals, moves, abilities. "
        "Cite by returning the tool's 'results' content; do not invent data."
    ),
)

# If you have a supervisor, include instruction like:
# - Route factual Pokédex queries to poke_rag_search.
# - Route image requests to sd tool node.
# - Route web when the local RAG returns empty.


In [8]:
from langchain_ollama import ChatOllama
from IPython.display import display, Markdown

llm = ChatOllama(model="gpt-oss:20b", temperature=0.2)

SUM_TMPL = """Answer the question concisely using ONLY the context.
If a count is asked, return just the number and a short confirmation line.
Avoid speculation.

Question: {q}

Context:
{ctx}
"""

def rag_answer(q: str, *, kind: str | None = None, k: int = 12):
    # 1) Try deterministic list for enums like generations/types/regions
    ql = q.lower()
    if any(x in ql for x in ["how many generations", "number of generations"]):
        res = poke_rag_search.invoke({"kind":"generation", "mode":"list", "view":"text", "k":50})
        lines = [ln for ln in res.get("text","").splitlines() if ln.strip().startswith("- ")]
        return f"There are {len(lines)} Pokémon generations (I–IX)."

    # 2) Generic retrieval → LLM summary
    res = poke_rag_search.invoke({
        "query": q,
        "kind": kind,
        "mode": "search",
        "view": "text",   # compact human-readable context
        "k": k
    })
    ctx = res.get("text") or "\n".join(
        f"- {h.get('name','')}: {h.get('snippet','')}"
        for h in res.get("results", [])
    ) or "No relevant context."
    ans = llm.invoke(SUM_TMPL.format(q=q, ctx=ctx[:6000])).content.strip()
    return ans



In [9]:
# Example:
#display(Markdown(rag_answer("what type of pokemon is slowpoke?")))
display(Markdown(rag_answer("what are all of the mythic pokemon")))

Flutter Mane

In [10]:
from IPython.display import display, Markdown, HTML

# Markdown table
res = poke_rag_search.invoke({
    "kind": "generation",
    "mode": "list",
    "view": "table",
    "k": 50,
})
display(Markdown(res["markdown"]))  # <- renders as a real table

res = poke_rag_search.invoke({
    "kind": "generation",
    "mode": "list",
    "view": "html",
    "k": 50,
})
display(HTML(res["html"]))          # <- renders HTML directly

# DataFrame view
res = poke_rag_search.invoke({
    "kind": "generation",
    "mode": "list",
    "view": "dataframe",
    "k": 50,
})
import pandas as pd
df = pd.DataFrame(res["rows"], columns=res["columns"])
display(df)

| Generation | Region |
|---|---|
| Generation I | Kanto |
| Generation II | Johto |
| Generation III | Hoenn |
| Generation IV | Sinnoh |
| Generation V | Unova |
| Generation VI | Kalos |
| Generation VII | Alola |
| Generation VIII | Galar |
| Generation IX | Paldea |

Generation,Region
Generation I,Kanto
Generation II,Johto
Generation III,Hoenn
Generation IV,Sinnoh
Generation V,Unova
Generation VI,Kalos
Generation VII,Alola
Generation VIII,Galar
Generation IX,Paldea


,Generation,Region
0,Generation I,Kanto
1,Generation II,Johto
2,Generation III,Hoenn
3,Generation IV,Sinnoh
4,Generation V,Unova
5,Generation VI,Kalos
6,Generation VII,Alola
7,Generation VIII,Galar
8,Generation IX,Paldea


In [19]:
# --- General-purpose Poké RAG Answerer (no per-species hacks) ---
import re
from typing import Optional, List, Dict, Any, Literal
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_ollama import ChatOllama

# ===== helpers =====
ROMAN = ["","I","II","III","IV","V","VI","VII","VIII","IX","X"]
def roman(n:int)->str: return ROMAN[n] if 0 <= n < len(ROMAN) else str(n)
def norm(s:str)->str: return re.sub(r"[^a-z0-9]+"," ", (s or "").lower()).strip()
def cap(s:str)->str:  return (s or "").replace("-", " ").title()

KINDS = {"generation","region","type","move","ability","species","pokemon"}
TYPE_SET = {"normal","fire","water","grass","electric","ice","fighting","poison","ground","flying",
            "psychic","bug","rock","ghost","dragon","dark","steel","fairy"}
REGION_SET = {"kanto","johto","hoenn","sinnoh","unova","kalos","alola","galar","paldea"}

LLM = ChatOllama(model="gpt-oss:20b", temperature=0.2)
SUM_TMPL = """Answer concisely using ONLY the context. If a count/list is asked, be precise.
Question: {q}

Context:
{ctx}
"""

def _cap(s:str) -> str:
    return (s or "").replace("-", " ").title()

def _fields_requested(q:str) -> list[str]:
    ql = q.lower()
    fields = []
    if "name" in ql:   fields.append("name")
    if "region" in ql: fields.append("region")
    if "type" in ql:   fields.append("types")
    if not fields:     fields = ["name"]  # sensible default
    return fields


# --- intent + signal extraction (generic, not species-specific) ---
def detect_intent(q:str)->str:
    ql = q.lower()
    if re.search(r"\bhow many\b|\bnumber of\b|\bcount\b", ql): return "count"
    if re.search(r"\b(list|show|what are(?: all)?|give me (?:all|every))\b", ql): return "list"
    # attribute/membership patterns: "what is/are the X of Y", "Y types?", "is Y mythical?"
    if re.search(r"\bis\b|\bare\b|\bof\b|\bdoes\b|\bdo\b", ql) and re.search(r"\btype|types|region|generation|ability|abilities|height|weight|size|power|accuracy|damage class|mythic", ql):
        return "attribute"
    if re.search(r"\bis .* (mythic|mythical|legendary)\b", ql): return "membership"
    return "qa"

def detect_kind_bias(q:str)->Optional[str]:
    ql = q.lower()
    for k in KINDS:
        if re.search(rf"\b{k}s?\b", ql): return k
    return None

def extract_types_filter(q:str)->List[str]:
    ql = q.lower()
    return [t.title() for t in TYPE_SET if re.search(rf"\b{t}\b", ql)]

def extract_region(q:str)->Optional[str]:
    ql = q.lower()
    for r in REGION_SET:
        if re.search(rf"\b{r}\b", ql): return r
    return None

def extract_attribute(q:str)->Optional[str]:
    # Map NL to metadata fields (broad, not species-specific)
    ql = q.lower()
    pairs = [
        (r"\btypes?\b", "types"),
        (r"\bregion\b", "region"),
        (r"\bgeneration\b", "generation"),
        (r"\babilities?\b", "abilities"),
        (r"\bheight\b|\bsize\b", "height_m"),
        (r"\bweight\b", "weight_kg"),
        (r"\bpower\b", "power"),
        (r"\baccuracy\b", "accuracy"),
        (r"\bdamage class\b", "damage_class"),
        (r"\b(mythis|mythic|mythical)\b", "is_mythical"),
        (r"\blegendary\b", "is_legendary"),
        (r"\bmoves?\b", "moves"),
        (r"\beffect\b|\bdoes\b|\bwhat happens\b", "effect_text"),
    ]
    for rx, field in pairs:
        if re.search(rx, ql): return field
    return None

def extract_target_name(q:str)->Optional[str]:
    # Generic extraction of the entity mention (species/pokemon/move/ability)
    # "what types is Slowpoke", "effect of Flamethrower", "ability of Snorlax", etc.
    m = re.search(r"\bof\s+([A-Za-z0-9' -]+)\??$", q.strip(), re.I) or \
        re.search(r"\b(?:is|are|for)\s+([A-Za-z0-9' -]+)\??$", q.strip(), re.I) or \
        re.search(r"\b([A-Za-z0-9' -]+)\s+(?:type|types|ability|abilities|effect)\??$", q.strip(), re.I)
    if m: 
        return m.group(1).strip()
    # bare entity question like "Charizard type?"
    m2 = re.search(r"^([A-Za-z0-9' -]+)\s+(?:type|types|effect|ability)\??$", q.strip(), re.I)
    return m2.group(1).strip() if m2 else None

# --- small deterministic operators over your metadata ---
def list_by_kind(kind:str)->List[Dict[str,Any]]:
    return poke_rag_search.invoke({"kind": kind, "mode": "list", "view": "pairs" if kind in {"species","pokemon"} else "names", "k": 200}).get("rows") or \
           poke_rag_search.invoke({"kind": kind, "mode": "list", "view": "names", "k": 200}).get("items", [])

MOVE_FIELDS = {"effect_text","power","accuracy","damage_class"}
SPEC_FIELDS = {"types","abilities","height_m","weight_kg","is_mythical","is_legendary","generation","region"}


def resolve_entity(name:str, prefer_attr:Optional[str]=None)->Dict[str,Any]:
    # decide search order from attribute
    order = []
    if prefer_attr in MOVE_FIELDS:
        order = ["move","ability","species","pokemon"]
    elif prefer_attr in SPEC_FIELDS:
        order = ["species","pokemon","move","ability"]
    else:
        order = ["species","pokemon","move","ability"]

    cands = []
    for k in order:
        res = poke_rag_search.invoke({"query": name, "kind": k, "mode": "search", "view": "raw", "k": 5})
        cands += res.get("results", [])

    if not cands:
        return {}

    def score(c):
        s = c.get("score", 0.0)
        exact = (norm(c.get("name")) == norm(name))
        return (1 if exact else 0, s)

    return sorted(cands, key=score, reverse=True)[0]

def read_attr_from_doc(doc: Dict[str, Any], field: str):
    import re
    m = (doc or {}).get("meta", {}) or {}
    # normalized helpers
    def _get(k, default=None): return m.get(k, default)

    if field == "effect_text":
        # prefer explicit meta
        eff = _get("effect_text")
        if eff: return eff
        # fallback: parse from snippet/text if your ETL concatenated "Effect: ..."
        blob = " ".join([doc.get("snippet") or "", doc.get("text") or ""]).strip()
        m1 = re.search(r"Effect:\s*(.+?)(?:\s*(?:Power|Accuracy|PP|$))", blob, flags=re.I)
        if m1:
            return m1.group(1).strip()
        return None

    if field in {"types","abilities","moves"}:
        vals = _get(field) or []
        return [v.replace("-", " ").title() for v in vals]

    if field in {"height_m","weight_kg","power","accuracy","damage_class","generation","region","is_mythical","is_legendary"}:
        return _get(field)

    return None

import re

def _norm(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", (s or "").lower()).strip()

def _find_move_doc_by_name(name: str) -> dict | None:
    """Exact-name first from the full local catalog; fallback to vector search."""
    nm = _norm(name)
    # list-mode: scan all moves deterministically
    lst = poke_rag_search.invoke({"kind": "move", "mode": "list", "view": "raw", "k": 5000})
    for d in lst.get("results", []):
        if _norm(d.get("name")) == nm:
            return d
    # fallback: vector
    raw = poke_rag_search.invoke({"query": name, "kind": "move", "mode": "search", "view": "raw", "k": 6})
    return (raw.get("results") or [None])[0]

def _read_move_effect(doc: dict) -> str | None:
    if not doc: return None
    m = (doc.get("meta") or {})
    eff = m.get("effect_text")
    if eff: return eff.strip()
    blob = " ".join([doc.get("snippet") or "", doc.get("text") or ""]).strip()
    if not blob: return None
    m1 = re.search(r"Effect:\s*(.+?)(?:\s*(?:Power|Accuracy|PP|Type|Class)\b|$)", blob, re.I)
    if m1: return m1.group(1).strip()
    m2 = re.search(r"(Has a .*? chance .*?\.|Burns the target\.|Deals .*? damage\.)", blob, re.I)
    return m2.group(1).strip() if m2 else None

def _read_move_stat(doc: dict, field: str):
    """
    field in {'power','accuracy','pp','damage_class'}
    Returns a normalized Python value or None if unknown.
    """
    if not doc: return None
    m = (doc.get("meta") or {})
    # 1) Prefer structured meta if present
    if field in {"power","accuracy","pp"} and m.get(field) is not None:
        return m[field]
    if field == "damage_class" and m.get(field):
        return str(m[field]).title()

    # 2) Parse from text/snippet if meta missing
    blob = " ".join([doc.get("snippet") or "", doc.get("text") or ""]).strip()
    if not blob: return None

    if field == "power":
        mm = re.search(r"\bPower:\s*(\d+)\b", blob, re.I)
        return int(mm.group(1)) if mm else None

    if field == "accuracy":
        mm = re.search(r"\bAccuracy:\s*(\d+)\b", blob, re.I)
        # PokéAPI sometimes uses null for “never misses”
        return int(mm.group(1)) if mm else "Never misses" if re.search(r"never misses", blob, re.I) else None

    if field == "pp":
        mm = re.search(r"\bPP:\s*(\d+)\b", blob, re.I)
        return int(mm.group(1)) if mm else None

    if field == "damage_class":
        mm = re.search(r"\b(Damage\s*Class|Class):\s*(Physical|Special|Status)\b", blob, re.I)
        return mm.group(2).title() if mm else None

    return None



# ===== Tool =====
class PokeQAArgs(BaseModel):
    question: str = Field(..., description="Any Pokémon question.")
    k: int = Field(12, ge=6, le=50)
    show_table: bool = Field(True)
    debug: bool = Field(False)

@tool("poke_rag_answer", args_schema=PokeQAArgs)
def poke_rag_answer(question:str, k:int=12, show_table:bool=True, debug:bool=False)->Dict[str,Any]:
    ''' Generates a simple response to a question about Pokemon '''
    def _norm(s: str) -> str:
        return re.sub(r"[^a-z0-9]+", " ", (s or "").lower()).strip()

    def _cap(s: str) -> str:
        return (s or "").replace("-", " ").title()

    def _strip_article(name: str) -> str:
        return re.sub(r"^(?:a|an|the)\s+", "", name.strip(), flags=re.I)

    def _find_entity_raw(name: str, kinds=("species","pokemon")) -> dict | None:
        """Try vector first, prefer exact-name; if not found, fall back to list-scan exact."""
        nm = _norm(name)
        best = None
        # vector search over requested kinds
        for k in kinds:
            raw = poke_rag_search.invoke({"query": name, "kind": k, "mode": "search", "view": "raw", "k": 6})
            pool = raw.get("results", []) or []
            exact = [d for d in pool if _norm(d.get("name")) == nm]
            if exact:
                return exact[0]
            best = best or (pool[0] if pool else None)
        if best:
            return best
        # fallback: list-mode exact name
        for k in kinds:
            lst = poke_rag_search.invoke({"kind": k, "mode": "list", "view": "raw", "k": 5000})
            for d in lst.get("results", []) or []:
                if _norm(d.get("name")) == nm:
                    return d
        return None

    def _read_shape_from_doc(doc: dict) -> str | None:
        """Return Title-cased shape from meta, if available."""
        if not doc:
            return None
        shp = ((doc.get("meta") or {}).get("shape") or "").strip()
        return _cap(shp) if shp else None

    # --- EARLY HANDLER: "what shape is X?" / "shape of X" / "X shape?" ---
    m = (
        re.search(r"\bwhat(?:'s| is)\s+the?\s*shape\s+(?:of\s+)?(.+?)\??$", question.strip(), re.I) or
        re.search(r"\bshape\s+of\s+(.+?)\??$", question.strip(), re.I) or
        re.search(r"\bwhat\s+shape\s+is\s+(.+?)\??$", question.strip(), re.I) or
        re.search(r"^(.+?)\s+shape\??$", question.strip(), re.I)
    )
    if m:
        target = _strip_article(m.group(1))
        doc = _find_entity_raw(target, kinds=("species","pokemon"))
        shp = _read_shape_from_doc(doc)
        if shp:
            name = _cap(doc.get("name") or target)
            return {"answer": f"{name} is {shp}.", "evidence": doc}
        # fall through to generic paths if not found

    intent = detect_intent(question)
    kind_bias = detect_kind_bias(question)
    attr = extract_attribute(question)
    target = extract_target_name(question)
    types_filter = extract_types_filter(question)
    region_filter = extract_region(question)
    is_mythic = bool(re.search(r"\bmythic(?:al)?\b", question.lower()))

    # --- STRUCTURED LIST: "all mythical/legendary ..." with field selection ---
    ql = question.lower().strip()
    want_myth = "mythic" in ql
    want_legend = "legendary" in ql
    # --- EARLY: "effect of <Move>" / "<Move> effect" (run this FIRST) ---
    mx = (
        re.search(r"\beffect\s+of\s+([A-Za-z0-9' -]+)\??$", ql, re.I)
        or re.search(r"\bwhat(?:'s| is)\s+the?\s*effect\s+of\s+([A-Za-z0-9' -]+)\??$", ql, re.I)
        or re.search(r"\bwhat(?:'s| is)\s+([A-Za-z0-9' -]+)\s+effect\??$", ql, re.I)
    )
    if mx:
        move_name = mx.group(1).strip()
        doc = _find_move_doc_by_name(move_name)   # exact-name list scan + vector fallback
        eff = _read_move_effect(doc)              # robust parser (below)
        if eff:
            return {"answer": eff,
                    "evidence": {"name": doc.get("name"), "kind": doc.get("kind"), "meta": doc.get("meta", {})}}

    # --- EARLY: move stats (power/accuracy/damage class/pp) ---
    mx = (
        re.search(r"\beffect\s+of\s+([A-Za-z0-9' -]+)\??$", ql, re.I)
        or re.search(r"\bwhat(?:'s| is)\s+the?\s*effect\s+of\s+([A-Za-z0-9' -]+)\??$", ql, re.I)
        or re.search(r"\bwhat(?:'s| is)\s+([A-Za-z0-9' -]+)\s+effect\??$", ql, re.I)
    )

    mx = (
        re.search(r"(?:what(?:'s| is)\s+)?(?:the\s+)?(power|accuracy|pp|damage(?:\s+)?class)\s+(?:of|for)\s+([A-Za-z0-9' -]+)\??$", ql, re.I)
        or re.search(r"^([A-Za-z0-9' -]+)\s+(power|accuracy|pp|damage(?:\s+)?class)\??$", ql, re.I)   # e.g., "Hydro Pump power?"
    )
    if mx:
        f1, f2 = mx.groups()
        # normalize which group is the field vs name
        field = f1.lower() if f1.lower() in {"power","accuracy","pp","damage class","damageclass"} else f2.lower()
        move  = f2 if field == f1.lower() else f1
        field = "damage_class" if "damage" in field else field  # unify

        doc = _find_move_doc_by_name(move)
        val = _read_move_stat(doc, field)
        name = (doc or {}).get("name") or move.title()

        if val is not None:
            if field == "accuracy" and isinstance(val, int):
                return {"answer": f"{name} accuracy: {val}%.", "evidence": {"kind":"move","name":name}}
            if field == "power":
                return {"answer": f"{name} power: {val}.", "evidence": {"kind":"move","name":name}}
            if field == "pp":
                return {"answer": f"{name} PP: {val}.", "evidence": {"kind":"move","name":name}}
            if field == "damage_class":
                return {"answer": f"{name} damage class: {val}.", "evidence": {"kind":"move","name":name}}
        # If we couldn’t parse it deterministically, fall through to generic QA.

    # If still nothing, fall through to generic QA

    if ("all" in ql or "list" in ql or "give me" in ql) and (want_myth or want_legend):
        # Pull ALL species deterministically (no vector search)
        raw = poke_rag_search.invoke({
            "kind":"species", "mode":"list", "view":"raw", "k":5000
        })
        # Build items (include fields you might ask for)
        items = []
        for h in raw.get("results", []):
            m = h.get("meta", {}) or {}
            if want_myth and not m.get("is_mythical"):   continue
            if want_legend and not m.get("is_legendary"): continue

            name = _cap(h.get("name"))
            gen  = (m.get("generation") or "").lower()
            region = (m.get("region") or m.get("main_region") or "").lower() or GEN_TO_REGION.get(gen, "")
            types_list = [_cap(t) for t in (m.get("types") or [])]

            items.append({
                "name": name,
                "region": _cap(region) if region else "—",
                "types": types_list,                 # keep as list; we join when rendering
                "generation": gen,                   # raw slug like 'generation-ii'
            })

        # Which columns did the user ask for?
        fields = _fields_requested(question)  # e.g. ["name","region"]

        # Render cells per field (this is the bug fix: nested comp)
        def cell(it, f):
            if f == "types":
                return ", ".join(it.get("types", [])) or "—"
            if f == "generation":
                g = it.get("generation", "")
                return ("Generation " + g.split("-", 1)[1].upper()) if g.startswith("generation-") else (_cap(g) or "—")
            return it.get(f, "—")

        rows = [[cell(it, f) for f in fields] for it in items]  # <-- FIXED

        # Optional: stable sort (by all columns)
        rows.sort(key=lambda r: tuple((x or "").lower() for x in r))

        # Build the markdown
        cols = [_cap(f) for f in fields]
        md = "| " + " | ".join(cols) + " |\n|" + "|".join(["---"]*len(cols)) + "|\n"
        md += "\n".join("| " + " | ".join(map(str, r)) + " |" for r in rows)

        answer = f"Found {len(items)} " + ("mythical" if want_myth else "legendary") + " Pokémon."
        return {"answer": answer, "table_markdown": md, "evidence": items}


    # 1) COUNT/LIST over enums → deterministic
    if intent in {"count","list"} and (kind_bias in {"generation","region","type"}):
        items = list_by_kind(kind_bias)
        if intent == "count":
            n = len(items)
            if kind_bias == "generation":
                ans = f"There are {n} Pokémon generations (I–{roman(n)})."
            else:
                ans = f"There are {n} Pokémon {kind_bias}s."
            return {"answer": ans, "evidence": items}
        # list
        names = [it["generation"] if kind_bias=="generation" else (it if isinstance(it,str) else it.get("name")) for it in items]
        ans = ", ".join(names)
        out = {"answer": ans, "evidence": items}
        if show_table and kind_bias=="generation":
            tbl = poke_rag_search.invoke({"kind":"generation","mode":"list","view":"table","k":50})
            out["table_markdown"] = tbl.get("markdown")
        return out

    # 2) ATTRIBUTE/MEMBERSHIP over an entity → resolve + read metadata
    if intent in {"attribute","membership"} and target:
        doc = resolve_entity(target, prefer_attr=attr)   # <-- pass attribute, not kind
        field = attr or ("is_mythical" if "mythic" in question.lower() else None)
        if doc and field is not None:
            val = read_attr_from_doc(doc, field)
            name = cap(doc.get("name"))
        if doc and field:
            val = read_attr_from_doc(doc, field)
            display_name = cap(doc.get("name"))
            if field == "types" and not val:
                # fallback: search the other (species/pokemon) bucket for types
                alt = resolve_entity(target, prefer_kind="pokemon" if doc.get("kind")=="species" else "species")
                val = read_attr_from_doc(alt, "types")
            if field in {"is_mythical","is_legendary"}:
                return {"answer": f"{display_name} is {'Yes' if val else 'No'}.", "evidence": doc}
            if field in {"height_m"} and val is not None:
                return {"answer": f"{display_name} is {val:.2f} m tall.", "evidence": doc}
            if field in {"weight_kg"} and val is not None:
                return {"answer": f"{display_name} weighs {val:.1f} kg.", "evidence": doc}
            if field in {"types","abilities","moves"} and val:
                cop = "is" if (field=="types" and len(val)==1) else "are"
                return {"answer": f"{display_name} {cop} " + " / ".join(val) + ".", "evidence": doc}
            if field in {"power","accuracy","damage_class"} and val is not None:
                return {"answer": f"{display_name} {field.replace('_',' ')}: {val}.", "evidence": doc}
            if field == "generation" and val:
                # translate 'generation-iv' to 'Generation IV'
                m = re.search(r"generation-(\w+)", str(val))
                return {"answer": f"{display_name} appears in Generation {m.group(1).upper() if m else str(val)}.", "evidence": doc}
            if field == "region":
                return {"answer": f"{display_name}: Region {cap(val)}.", "evidence": doc}
            if field == "effect_text" and val:
                return {"answer": val, "evidence": doc}
        # If attribute not found, fall through to QA

    # 3) LIST filtered entities (species/move/ability) → search + format
    if intent == "list" and (kind_bias in {"species","pokemon","move","ability"} or types_filter or region_filter or is_mythic):
        res = poke_rag_search.invoke({
            "query": question,
            "kind": kind_bias,
            "types": types_filter or None,
            "region": region_filter,
            "mythical_only": is_mythic,
            "require_all_types": True,
            "mode": "search",
            "view": "pairs" if (kind_bias in {"species","pokemon"} or not kind_bias) else "names",
            "k": k
        })
        names = [r["name"] for r in res.get("rows", [])] or res.get("items") or []
        ans = "None found." if not names else (", ".join(names) if len(names)<=10 else f"{', '.join(names[:10])}, and {len(names)-10} more.")
        return {"answer": ans, "evidence": res}

    # 4) GENERAL QA → vector search then LLM summarize (kind bias helps retrieval)
    res = poke_rag_search.invoke({
        "query": question,
        "kind": kind_bias,
        "mode": "search",
        "view": "text",
        "k": k
    })
    ctx = res.get("text") or "\n".join(
        f"- {h.get('name','')}: {h.get('snippet','')}" for h in res.get("results", [])
    ) or "No relevant context."
    ans = LLM.invoke(SUM_TMPL.format(q=question, ctx=ctx[:6000])).content.strip()
    out = {"answer": ans, "evidence": res}
    if debug: out["debug"] = {"intent": intent, "kind_bias": kind_bias, "attribute": attr, "target": target}
    return out



In [20]:
from IPython.display import Markdown, display

def ask_poke(q, **kwargs):
    res = poke_rag_answer.invoke({"question": q, **kwargs})
    display(Markdown(f"**Q:** {q}\n\n**A:** {res['answer']}"))
    return res

In [21]:
res = poke_rag_answer.invoke({"question":"what shape is a slowpoke"})
from IPython.display import Markdown, display
display(Markdown(f"**A:** {res['answer']}"))
#display(Markdown(res["table_markdown"]))


**A:** Slowpoke is Quadruped.

In [22]:
#result = ask_poke("how many pokemon generations are there?")
#result = ask_poke("give me the names and regions of all mythical pokemon")
#ask_poke("what is the effect of Flamethrower?")
#ask_poke("what is the power of Hydro Pump?")
#ask_poke("Thunderbolt accuracy?")
#ask_poke("Shadow Ball damage class?")
ask_poke("What are all of the possible shape values?")

**Q:** What are all of the possible shape values?

**A:** Normal, Forecast, Unknown, Shadow

{'answer': 'Normal, Forecast, Unknown, Shadow',
 'evidence': {'text': '- Normal\n- Forecast\n- Staryu — Water\n- Togepi — Fairy\n- Unown — Psychic\n- Castform — Normal\n- Palkia — Water, Dragon\n- Gigalith — Rock\n- Flabebe — Fairy\n- Doublade — Steel, Ghost\n- Unknown\n- Shadow'}}

In [18]:
import inspect
print(inspect.signature(PokeRAG._filter))   # ... shape=None
print(inspect.signature(PokeRAG.search))    # ... shape=None

# Tool path:
poke_rag_search.invoke({"query":"slowpoke","kind":"species","mode":"search","view":"pairs","k":1})
# QA path:
ask_poke("what shape is a slowpoke?")

(self, d: Dict[str, Any], *, kind: Optional[str], mythical_only: bool, types: Optional[List[str]], region: Optional[str], generation: Optional[str], shape: Optional[str]) -> bool
(self, query: str, k: int = 8, *, kind: Optional[str] = None, mythical_only: bool = False, types: Optional[List[str]] = None, region: Optional[str] = None, generation: Optional[str] = None, shape: Optional[str] = None) -> List[Dict[str, Any]]


**Q:** what shape is a slowpoke?

**A:** The context does not provide shape information for Slowpoke.

{'answer': 'The context does not provide shape information for Slowpoke.',
 'evidence': {'text': '- Raticate — Normal\n- Diglett — Ground\n- Slowpoke — Water, Psychic\n- Slowbro — Water, Psychic\n- Aipom — Normal\n- Slowking — Water, Psychic\n- Piloswine — Ice, Ground\n- Slakoth — Normal\n- Slaking — Normal\n- Spinda — Normal\n- Ambipom — Normal\n- Hippopotas — Ground'}}